# Instructions
Before you start this lesson you need to set up Google Colab a bit more:

1. Click the Runtime > Change runtime type menu option at the top of the page:
   ![Screenshot of change runtime type menu](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-change-runtime-type-menu.png?raw=true)
2. Then select "T4 GPU" under the hardware accelerator section and click "Save"
   ![Change runtime type popup screenshot](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-change-runtime-type-popup.png?raw=true)
3. Finally Click the "Run all" button at the top of the page:
   ![Screenshot showing run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/run-all-button.png?raw=true)
4. If you completed these instructions correctly you should see this message a few inches below on this page:
   ![Screenshot of image showing pytorch setup was successful](https://github.com/catvec/machine-learning-camp/blob/main/assets/neural-networks/colab-pytorch-accel-found.png?raw=true)
   If you don't see this message please verify you completed steps 1 and 2 correctly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, interactive_output
import ipywidgets as widgets
import IPython.display as display
import torch
from torch import nn
from torch.utils.data import Dataset, random_split
import graphviz
import plotly.graph_objects as go
from pprint import pprint

In [ ]:
# @title
if torch.accelerator.is_available():
    print("Excellent, you followed the setup steps correctly")
else:
    print("Whoops! It looks like Google Colab isn't setup correctly, please review the setup instructions above")

# What are Large Language Models (LLMS)?

Large Language Models are machine learning models built to consume and produce large volumes of text. You've probably heard quite a bit about LLMs in the past few years with chatbots like ChatGPT and Claude. Many are built on architecture called a **transformer**, which build off the neural networks you just learned about. LLMs are turning out to be good **proxies** for *reasoning about sequences* such as language, code, DNA, music, etc.

In this unit, we will learn how they work.


# Why?

Why were LLMs designed in the first place? How are we using them now?

In the mid-20th century, computing machines allowed for the modeling of language at complexity and scale previously unseen. At the same time, the world was breaking out into war. Encrypting messages to keep their contents secret from unwatned observers became a focus of those involved, and large efforts took part to crack those codes.

Alan Turing, an English mathematician and computer scientist, joined a government program to crack a code called Enigma that was used by the Nazis in 1939. To break it, they needed to figure out what made a sequence of symbols random vs meaningful. Turing played a crucial role in developing the math to model language, and develop the machines to mechanically search for meaning in randomness.

Throughout the following decades, work continued on understanding languge in other contexts. With computation getting more powerful, we were able to start crunching the numbers and model the statistics of language. In the 1940s, people such as C. E. Shannon were working on the field of **Natrual Lanugage Processing** (NLP) and trying to solve a fundamental problem - can we predict the next letter in a sequence of English text based on the preceeding letters?

As the Cold War progressed in the 1950's, machine translation become of particular importance. The Georgetown-IBM experiment became the first machine translation model aimed at translating Russian scientific literature to English. ELIZA in 1966 used keyword pattern matching to simulate a therapist, showing how much can be faked with surface patterns.

As research continued, people tried to approach problems such as speech recognition and started to move from representing grammer as rules to statistically discover rules in *corpa* (large bodies of text collected as a representative sample).

Nearing the turn of the century, **n-gram** was the buzzword. Researchers built statistical models to predict the next word based on the previous N words (typically 2-5). These models powered spell checks, speech recognition, early Google Translate, and autocomplete. These were limited by lacking any memory beyond the 2-5 previous words.

Neural networks, as you covered in the previous lesson, took off in the early 2010s. This is where we will continue our lesson, and we will learn how we got from neural networks to the modern LLM.

Today, language models are applied in many creative ways. Of course, LLM-based chatbots are useful in ways you've seen before. They are proving to be especially good at reading and writing structured text so products like GitHut Copilot and Claude Code are being used extensively to write code.

In biology, ESMFold and AlphaFold are transformer models trained on protein structures instead of human language, since they are also strings of symbols encoding meaning. This work has effectively "solved" a key problem in biology, which is understanding the structure of a protein from its enciding sequence (eg. amino acids, RNA). Similarly, they have been useful in biochemistry, helping us generate and screen potential antibodies when searching for new medicines.

LLMs are generating novel material candidates, controlling robotic chemistry platforms, iterating experiments, and summarizing large legal texts.

As with all technology, all of these come at costs that are important to keep in mind.


In [ ]:
# @title
# Display multiple questions with true/false buttons, then check if answers are correct

questions_data = [
    {
        "question": "Alan Turing joined a government program to help crack the Enigma code used by the Nazis.",
        "answer": "True",
        "hint": "Turing worked on modeling language mathematically to distinguish random sequences from meaningful ones, helping break Enigma in 1939."
    },
    {
        "question": "C. E. Shannon's work in the 1940s explored whether we could predict the next letter in English text based on the letters before it.",
        "answer": "True",
        "hint": "This was a foundational problem in early Natural Language Processing (NLP) research."
    },
    {
        "question": "The Georgetown-IBM experiment was designed to translate English scientific literature into Russian.",
        "answer": "False",
        "hint": "It actually went the other direction: the experiment translated Russian scientific literature into English."
    },
    {
        "question": "ELIZA simulated a therapist using deep neural networks trained on large corpora of text.",
        "answer": "False",
        "hint": "ELIZA (1966) used simple keyword pattern matching, not neural networks, showing how much can be faked with surface patterns."
    },
    {
        "question": "AlphaFold and ESMFold are transformer models trained on human language.",
        "answer": "False",
        "hint": "They are transformer models trained on protein structures (like amino acid sequences) instead of human language."
    },
]

def make_question_widget(question, answer, hint):
    question_label = widgets.Label(value=question)
    true_button = widgets.Button(description="True")
    false_button = widgets.Button(description="False")
    answer_result = widgets.Label()
    hint_label = widgets.Label()

    def check_answer(event):
        if event.description == answer:
            answer_result.value = "✅ Correct!"
            hint_label.value = ""
        else:
            answer_result.value = "❌ Incorrect :("
            hint_label.value = hint

    true_button.on_click(check_answer)
    false_button.on_click(check_answer)

    return widgets.VBox([
        question_label,
        widgets.HBox([true_button, false_button]),
        answer_result,
        hint_label,
    ])

all_questions = widgets.VBox([
    make_question_widget(q["question"], q["answer"], q["hint"])
    for q in questions_data
])

display.display(all_questions)

In [ ]:
# @title
# Put each event in the correct order using the dropdowns, then check your answer
items = [
    "Turing works to crack the Enigma code",
    "Shannon asks whether we can predict the next letter in English text",
    "The Georgetown-IBM experiment translates Russian science writing into English",
    "ELIZA fakes being a therapist using keyword matching",
    "n-gram models predict the next word from the last few words",
    "Neural networks take off and start replacing statistical language models",
]

correct_order = items

slot_labels = [widgets.Label(value=f"{i+1}") for i in range(len(items))]
slot_dropdowns = [widgets.Dropdown(options=items, description="") for i in range(len(items))]
check_button = widgets.Button(description="Check order")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    chosen_order = [dropdown.value for dropdown in slot_dropdowns]
    if chosen_order == correct_order:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = "Think about which happened during World War 2, and which happened after neural networks became popular"

check_button.on_click(check_answer)

rows = [widgets.HBox([slot_labels[i], slot_dropdowns[i]]) for i in range(len(items))]

display.display(widgets.VBox([
    widgets.Label(value="Put these events in the order they actually happened:"),
    *rows,
    check_button,
    answer_result,
    hint_label,
]))

# Recurrent Neural Networks
### Sequential data
If we want to make a machine that can read and write language, we need to start looking at language as data. A ciritcal aspect of the data of language is that it is **sequential.**
***

This sentence means nothing if the words are not in the correct order.

*are correct if in means not nothing order sentence the the This words.*
***

The meaning of words can change based on the other words around them. For example: `The bank by the *river*` vs `The bank by the *post office*`.

The neural network's you learned about last lesson are called ***feedforward*** networks. As you saw with the weather example, execution starts with fixed inputs (ex. temp, humidity, cloud cover), each layer transforms the data and passes it to the next, and eventually the final layer's output is collected.

What happens if we try to apply feedforward networks to sequential data? What would our training data look like? Could it even work if the input is a sentence of variable length?

- One problem is that feedforward networks need inputs of **fixed length**
  - Sequential data, especially language, is not always


To deal with this shortcoming, there was a need for NNs that could retain information and carry it forward.

# Recurrent Neural Networks

The firsrt step along the path from NNs to LLMs is a type of neural network called the **recurrent neural network.** We use the term RNN to refer to a family of algorithms that use neural network architecture to process sequences, instead of individual inputs.

### Sequential data
If we want to make a machine that can read and write language, we need to start looking at language as data. A ciritcal aspect of the data of language is that it is **sequential.**
***

This sentence means nothing if the words are not in the correct order.

*are correct if in means not nothing order sentence the the This words.*
***

The meaning of words can change based on the other words around them. For example: `The bank by the *river*` vs `The bank by the *post office*`. To deal with this, there was a need for NNs that could retain information and carry it forward.

You last saw a neuron represented like this:
***
$$
y = \tanh(w \cdot x + b)
$$
***
An RNN uses the same equation, but with an extra input called a *hidden state*.

- **New term: *timestep*** - one step in processing a sequence. For example, while processing the string "Hello world", the first timestep would be passing the model "Hello" and the second timestep would be processing "world."

**New term: *token*** - one chunk of a sequence that a model processes at a single timestep. In the RNN examples in this notebook, each token is a single character. In real LLMs, tokens are usually closer to word sized, for example `the cat sat` would be 3 tokens. From here on we will use the word token instead of character or word, since it covers both.

At each timestep an RNN first does one "forward pass" through the neuron equation and produces a *hidden state*. Intuitively, this is basically just the output of the last layer of an NN. But RNNs use it as input in the same neuron in the next timestep. Here is our new equation:
***
$$
h_t = \tanh(W \cdot x_t + U \cdot h_{t-1} + b)
$$
***
What does this all mean? When we use the subscript $_t$ it means "at the timestep $t$."

We've changed out $h$ to $h_t$ to show it is an equation for $h$ at the current timestep $t$. Inside our activation function, we multiply the weights by our input as we did before, but we add a new term $U\cdot h_(t-1)$. Here $U$ is a new matrix of weights similar to $W$ but applied to the memory instead of the input. The memory here is represented by $h_(t-1)$ which means the $h$ value calculated by the equation at the previous timestep, aka the *hidden state*. $b$ is the same bias as before.
***

## Limitations
RNN's represent memory by using the hidden state generated on the previous pass. This is a single, fixed-length vector, and on every pass, it takes on a completely new value.

Imagine you are reading the book, and attempting to summarize the entire thing in one sentence. After each chapter you read, you rewrite the sentence to include the new chapter. If you want to remember what happened in the first chapter, you'll get a much better picture reading the first version of the sentence than the one you've got at the end of the book.

As sequences get longer, the way RNN's store memory starts to break down. The hidden state stays the same fixed size no matter how many timesteps you feed it, so more data is necessarily lost the longer the sequence gets. This is called the **bottleneck problem** and is a critical issue with RNN architecture - there is only one small container for arbitrarily large context.

***
The other problem involves the concept of **stochastic gradient descent** - review the concept in the Neural Networks notebook before continuing if needed.

When training any neural network, the "error signal" must travel backwards through the network. This is a confusing concept - the execution of the model or program never "runs backwards." What this really means is that after producing an output and calculating the error at the final step, that error is propagated backward through every layer to compute how much each weight contributed to it, and then all the weights are updated together, in a single pass.

So the error must travel backwards through the network to reach the weights that processed the very first inputs. For RNN's, we can think of them as very deep networks, with one "layer" per timestep. So the error must travel through each timestep to reach the weights that processed the first inputs.

At each timestep, the gradient gets multiplied by the **derivative** of the $\tanh$ function, which is always between 0 and 1 (and if derivatives are new to you, think of the derivative of a function as its **slope**). If you multiply something by a value less than 1 enough times in a row, it shrinks down to close to 0. By the time the gradient has reached the weights at timestep 1 of a very long sequence, it is essentially 0, so the weights that processed early inputs barely get updated. This is called the **vanishing gradient** problem.

The opposite can also happen: if the values being multiplied at each step are greater than 1, the gradient can grow instead of shrink, blowing up to a huge number by the time it reaches the early timesteps. This is called the **exploding gradient** problem, and it's why RNN training sometimes produces weights that suddenly spike to extreme values.

***

**Summary:** Bottlenecking occurs because the memory is fixed in size, and loses information each time it is updated. This is a design problem. Vanishing and exploding gradients occur because the model must update itself at each timestep, and the gradient is repeatedly multiplied by small, or large, values, causing it to shrink toward 0, or grow explosively, by the time it reaches early weights. This is a training problem.


Play around with the visualization below. This shows how much of the previous characters are influencing the prediction of the next character. Observe the range at which characters influence each other, and how it changes over the sequence.

In [ ]:
# @title
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import numpy as np
import IPython.display as display
import ipywidgets as widgets
from ipywidgets import interact

# A short, repetitive sentence so the RNN has a real job:
# predicting "mat" requires remembering "the" from much earlier
TEXT = "the cat sat on the mat the cat sat on the mat"

chars = sorted(set(TEXT))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
VOCAB = len(chars)
SEQ = torch.tensor([c2i[c] for c in TEXT], dtype=torch.long)

class CharRNN(nn.Module):
    def __init__(self, vocab, hidden=48):
        super().__init__()
        self.embed = nn.Embedding(vocab, hidden)
        self.rnn = nn.RNN(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, embeds):
        out, _ = self.rnn(embeds)
        return self.fc(out)

model = CharRNN(VOCAB)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(600):
    x_embed = model.embed(SEQ[:-1].unsqueeze(0))
    logits = model(x_embed)
    loss = loss_fn(logits.squeeze(0), SEQ[1:])
    opt.zero_grad(); loss.backward(); opt.step()

T = len(SEQ) - 1  # number of input positions
labels = [TEXT[i] for i in range(T)]

def get_connectivity(target_pos):
    """
    For a chosen target position, compute how strongly each
    earlier input character influenced that prediction.
    This is the gradient magnitude of the output w.r.t. each input embedding.
    """
    embeds = model.embed(SEQ[:-1].unsqueeze(0)).detach().requires_grad_(True)
    logits = model(embeds)
    score = logits[0, target_pos].max()
    score.backward()
    grad = embeds.grad[0].norm(dim=-1).detach().numpy()
    grad[target_pos + 1:] = 0.0
    mx = grad.max()
    return grad / mx if mx > 0 else grad

header = widgets.HTML("""
<div style="font-family:monospace; font-size:13px; line-height:1.7; margin-bottom:8px">
  The RNN was trained to predict the <b>next character</b> at each position.<br>
  Use the slider to pick a position. The heatmap shows how much each
  <b>earlier</b> character influenced that prediction.<br>
  <span style="color:#E53935">■</span> red box = position you selected &nbsp;|&nbsp;
  <span style="color:#1565C0">■</span> dark blue = strong influence &nbsp;|&nbsp;
  light = weak or none
</div>
""")
display.display(header)

def update(target_pos):
    """Move the slider to pick a target position."""
    connectivity = get_connectivity(target_pos)
    predicted_char = i2c[int(model(
        model.embed(SEQ[:-1].unsqueeze(0)).detach()
    ).squeeze(0)[target_pos].argmax())]

    fig, ax = plt.subplots(figsize=(max(6, T * 0.5), 2.2))

    z = np.array(connectivity).reshape(1, -1)
    im = ax.imshow(
        z, cmap="Blues", vmin=0, vmax=1,
        aspect="auto", interpolation="nearest"
    )

    ax.set_xticks(range(T))
    ax.set_xticklabels(labels, family="monospace", fontsize=11)
    ax.set_yticks([0])
    ax.set_yticklabels(["influence"], family="monospace", fontsize=10)

    rect = patches.Rectangle(
        (target_pos - 0.5, -0.5), 1, 1,
        linewidth=3, edgecolor="#E53935", facecolor="none"
    )
    ax.add_patch(rect)

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("strength", family="monospace", fontsize=10)

    ax.set_title(
        f"Predicting '{predicted_char}' at position {target_pos}  "
        f"('{TEXT[target_pos]}'→'{predicted_char}')  "
        f"— darker = stronger influence on this prediction",
        family="monospace", fontsize=11
    )
    ax.set_xlabel("input character position", family="monospace", fontsize=10)

    plt.tight_layout()
    plt.show()

interact(
    update,
    target_pos=widgets.IntSlider(
        min=0, max=T - 1, value=T // 2, step=1,
        description="target pos",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="520px"),
    ),
)

In [ ]:
# @title
# Display question and answer choices as buttons, then check if the chosen one is correct
question = "Based on the heatmap above, what happens to an RNN's attention on early characters as the sequence gets longer?"
choices = [
    "It stays exactly the same no matter how far back the character was",
    "It gets stronger the further back the character is",
    "It resets to zero at the start of every new word",
    "It fades out, later predictions depend mostly on only the most recent few characters",
]
answer = choices[3]
hint = "This is exactly the bottleneck problem, information gets compressed and lost the further back it is"

question_label = widgets.Label(value=question)
choice_buttons = [widgets.Button(description=choice, layout=widgets.Layout(width="600px")) for choice in choices]
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if event.description == answer:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

for button in choice_buttons:
    button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    *choice_buttons,
    answer_result,
    hint_label,
]))

In [ ]:
# @title
# Display multiple questions with true/false buttons, then check if answers are correct

questions_data = [
    {
        "question": "In an RNN, the meaning of a word can depend on the other words around it, which is part of why sequential order matters.",
        "answer": "True",
        "hint": "The 'bank by the river' vs 'bank by the post office' example shows how surrounding words change meaning, which is why RNNs process sequences instead of individual inputs."
    },
    {
        "question": "A 'timestep' refers to one full pass through the entire training dataset.",
        "answer": "False",
        "hint": "A timestep is one step in processing a sequence, such as feeding in a single token like 'Hello' before moving on to the next token."
    },
    {
        "question": "In the RNN equation, the hidden state from the previous timestep is combined with the current input to help compute the new hidden state.",
        "answer": "True",
        "hint": "The equation h_t = tanh(W·x_t + U·h_(t-1) + b) uses h_(t-1), the previous hidden state, as memory carried into the current timestep."
    },
    {
        "question": "The bottleneck problem happens because RNNs store memory in a hidden state that stays a fixed size no matter how long the sequence gets.",
        "answer": "True",
        "hint": "Since the hidden state size never grows, longer sequences necessarily lose more information as new data overwrites old memory."
    },
    {
        "question": "The vanishing gradient problem occurs because the gradient is repeatedly multiplied by values greater than 1 as it travels backward through timesteps.",
        "answer": "False",
        "hint": "Vanishing gradients happen when the gradient is multiplied by values between 0 and 1 at each timestep, shrinking it toward 0. >1 causes exploding gradient.",
    },
]

def make_question_widget(question, answer, hint):
    question_label = widgets.Label(value=question)
    true_button = widgets.Button(description="True")
    false_button = widgets.Button(description="False")
    answer_result = widgets.Label()
    hint_label = widgets.Label()

    def check_answer(event):
        if event.description == answer:
            answer_result.value = "✅ Correct!"
            hint_label.value = ""
        else:
            answer_result.value = "❌ Incorrect :("
            hint_label.value = hint

    true_button.on_click(check_answer)
    false_button.on_click(check_answer)

    return widgets.VBox([
        question_label,
        widgets.HBox([true_button, false_button]),
        answer_result,
        hint_label,
    ])

all_questions = widgets.VBox([
    make_question_widget(q["question"], q["answer"], q["hint"])
    for q in questions_data
])

display.display(all_questions)




## Check in
Last lesson you took a network and passed it some weather data, and it output a probability of the weather being good. RNNs are passed one part of a sequence, output some vector, then the model is run again, this time with the next part of the sequence and the output vector of the previous pass. **Key takeaway:** Recurrent neural networks retain memories across a sequence, but that memory is a single fixed-size vector that get rewritten at every step, so early information fades quickly and long sequences are hard to learn from.

# LSTM and GRU

Reearchers recognized the vanishing gradient problem in RNN's early on. **Long short-term memory** (LSTM) models were introduced in 1997 as a way to address this problem. These add a second vector called a *cell state* that runs alongside the hidden state. The hidden state keeps getting updated as before, but the new cell state is only updated according to a *gate* mechanism. This mechanism decides what to forget from the previous cell state, what new information to write in, and what the current timestep's output should be. LSTM architecture gave RNN models an explicit and controllable memory that can hold information over much longer sequences.

**Gated recurrent units** (GRUs) were another attempt to solve this problem, introduced in 2014 as simlified versions of the LSTM models. It produces similar results but can be trained much faster.

Both architectures improved performance on long sequences and made real impacts in the world through the 2010s. But these models only push the bottleneck problem further down the road, since memory was still being compressed into fixed-length vectors.



# RNN's in practice - Encoder-Decoder

Despite the limitations above, RNNs proved to be very useful in real world applications. One of the most common was **machine translation**, using machines to automatically translate text between languages.

A regular RNN won't work here, because the input and output sequences will have different numbers of words and characters.

Researchers found a solution in chaining two RNNs together. The first RNN is called an **encoder.** The encoder accepts the input sequence, one token at a time, and builds up a hidden state. It produces no output, and when it finishes processing the sequence we are left with it's internal hidden state, which now represents a compression of everything it just read. This hidden state is then passed to another RNN called the **decoder.** Instead of starting with a random sequence of numbers for its memory, as a regular RNN would do, the decoder uses the encoder's hidden state as it's initial memory. Then, it generates an output sequence, one token at a time.

Imagine one person reads a text in English, then summarizes in on a notecard. Then, another person reads the notecard then writes the text in French, referring back to the notecard as the go.

This architecture is called **seq2seq** (sequence to sequence) and was a major breakthrough when introduced in 2014. But once again, we are compressing information down to a fixed-length vector. No matter how long the input sequence, the encoder still has to fit everything into it's memory. The bottleneck problem is still present. For long inputs, critical information may be lost.

Recall the visualization above. You were able to see the limitations of RNNs, how influence from information earlier in the sequence fades quickly and is completely forgotten after a few timesteps.RNN's, no matter the improvmenets made, still only process tokens one at a time, left to right, and compress information to pass it between parts of the network. This is a fundamental problem that **attention** was designed to solve.

In [ ]:
# @title
items = [

    "The decoder generates the output sequence one token at a time",
    "The encoder reads the input sequence one token at a time",
    "The decoder uses that hidden state as its starting memory",
    "The encoder finishes and hands off its hidden state to the decoder",
]

correct_order = [
    "The encoder reads the input sequence one token at a time",
    "The encoder finishes and hands off its hidden state to the decoder",
    "The decoder uses that hidden state as its starting memory",
    "The decoder generates the output sequence one token at a time",
]

slot_labels = [widgets.Label(value=f"{i+1}") for i in range(len(items))]
slot_dropdowns = [widgets.Dropdown(options=items, description="") for i in range(len(items))]
check_button = widgets.Button(description="Check order")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    chosen_order = [dropdown.value for dropdown in slot_dropdowns]
    if chosen_order == correct_order:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = "Think about who has to finish reading before anyone can start writing"

check_button.on_click(check_answer)

rows = [widgets.HBox([slot_labels[i], slot_dropdowns[i]]) for i in range(len(items))]

display.display(widgets.VBox([
    widgets.Label(value="Put the seq2seq steps in the order they happen:"),
    *rows,
    check_button,
    answer_result,
    hint_label,
]))

# Attention

Before we continue, let's define a couple terms.

**Token:** You may have heard this word when people talk about AI usage. A token is *one chunk of a sequence.* In many RNN's, including the ones we saw above, each character is one token. In LLMs, tokens are roughly word-sized. For example, `the cat sat` would be 3 tokens.


Even a plain RNN does carry information forward - an early difference will ripple through the hidden state at every later position. But it has to travel through every single timestep to get there, getting compressed and mixed with everything else along the way. By the time you are 40 tokens into a paragraph, that early information is faint at best.

Attention throws out the idea that a model has to squeeze everything into one hidden state that gets rewritten at every step. Instead, at every timestep, the model gets to look back directly at every earlier token and decide for itself which ones actually matter right now.

Go back and think about the book summary analogy from the Limitations section. RNNs are like rewriting one sentence of notes after every chapter. Attention is like keeping the whole book open on the desk, and flipping directly back to whatever page you actually need.

## Query, Key, Value

To decide which earlier tokens matter, attention borrows an idea from something you may have already done, searching a filing cabinet.

- **Query**: what the current token is looking for
- **Key**: a label on each earlier token describing what it contains
- **Value**: the actual information an earlier token has to offer, if it gets picked

Every token gets turned into all three of these using their own set of learned weights, just like the weights in the neurons from last lesson. The model compares the current token's query against every earlier token's key, using a dot product, which gives a score for how well they match. The higher the score, the more that earlier token's value gets used.

Those scores get passed through a **softmax** function, which squashes a list of numbers so they are all positive and add up to 1, basically turning the scores into a set of percentages. Those percentages are the attention weights. The output for the current token is just a weighted average of every earlier token's value, weighted by how much attention it is paying to each one.

Here is the actual equation used, do not worry about memorizing it, just get a feel for the pieces:
***
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$
***
$Q$, $K$, and $V$ are matrices holding the query, key, and value for every token in the sequence at once. $QK^T$ computes the dot product score between every pair of tokens in one matrix multiplication. Dividing by $\sqrt{d_k}$ just keeps the scores from getting too large as the size of the vectors grows, this is a detail you do not need to worry about.

In [ ]:
# @title
question = "What does the softmax step in attention actually do?"
choices = [
    "Trains the query and key weights",
    "Deletes tokens the model does not need",
    "Turns the raw similarity scores into a set of weights that all add up to 1",
    "Converts characters into tokens",
]
answer = choices[2]
hint = "Softmax takes raw numbers and turns them into something more useful, in this case a set of weights that add up to 1"

question_label = widgets.Label(value=question)
choice_buttons = [widgets.Button(description=choice, layout=widgets.Layout(width="600px")) for choice in choices]
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if event.description == answer:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

for button in choice_buttons:
    button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    *choice_buttons,
    answer_result,
    hint_label,
]))

## One Rule: No Peeking at the Future

If a model is trying to predict the next token, it obviously cannot be allowed to look at tokens that have not happened yet. So attention used for this kind of prediction gets a mask applied to it, tokens are only allowed to attend to themselves and whatever came before them. This is usually called **causal** attention, or masked self attention. You will see this exact idea in the code below.

Attention that looks in only one direction like this, at everything so far, is called self attention when it happens within a single sequence. If you remember the encoder and decoder from the last section, attention can also be used to let the decoder look directly at every position in the encoder's output, instead of just its final hidden state. That version is usually called cross attention.

In [ ]:
# @title
question = "A model generating text one token at a time is allowed to use self attention to look ahead at tokens it has not generated yet."
answer = "False"
hint = "This is why it is called causal attention, a token can only attend to itself and whatever came before it"

question_label = widgets.Label(value=question)
true_button = widgets.Button(description="True")
false_button = widgets.Button(description="False")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if event.description == answer:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

true_button.on_click(check_answer)
false_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    widgets.HBox([true_button, false_button]),
    answer_result,
    hint_label,
]))

Now, we are going to train a tiny attention-based model on the exact same sentence as the RNN example earlier, and look at what it is paying attention to.

In [ ]:
# @title
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import numpy as np
import IPython.display as display
import ipywidgets as widgets
from ipywidgets import interact

TEXT = "the cat sat on the mat the cat sat on the mat"

chars = sorted(set(TEXT))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
VOCAB = len(chars)
SEQ = torch.tensor([c2i[c] for c in TEXT], dtype=torch.long)

class CharAttention(nn.Module):
    def __init__(self, vocab, hidden=48):
        super().__init__()
        self.embed = nn.Embedding(vocab, hidden)
        self.query = nn.Linear(hidden, hidden)
        self.key = nn.Linear(hidden, hidden)
        self.value = nn.Linear(hidden, hidden)
        self.fc = nn.Linear(hidden, vocab)
        self.hidden = hidden

    def forward(self, embeds):
        q = self.query(embeds)
        k = self.key(embeds)
        v = self.value(embeds)

        scores = q @ k.transpose(-2, -1) / (self.hidden ** 0.5)

        seq_len = embeds.shape[1]
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        attended = weights @ v
        return self.fc(attended), weights

model = CharAttention(VOCAB)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(600):
    x_embed = model.embed(SEQ[:-1].unsqueeze(0))
    logits, _ = model(x_embed)
    loss = loss_fn(logits.squeeze(0), SEQ[1:])
    opt.zero_grad(); loss.backward(); opt.step()

T = len(SEQ) - 1
labels = [TEXT[i] for i in range(T)]

def get_attention_weights(target_pos):
    x_embed = model.embed(SEQ[:-1].unsqueeze(0))
    with torch.no_grad():
        _, weights = model(x_embed)
    w = weights[0, target_pos].numpy().copy()
    w[target_pos + 1:] = 0.0
    return w

header = widgets.HTML("""
<div style="font-family:monospace; font-size:13px; line-height:1.7; margin-bottom:8px">
  This model predicts the <b>next character</b> using self attention instead of a hidden state.<br>
  Use the slider to pick a position. The heatmap shows the actual attention weight
  it assigned to each <b>earlier</b> character when making that prediction.<br>
  <span style="color:#E53935">■</span> red box = position you selected &nbsp;|&nbsp;
  <span style="color:#1565C0">■</span> dark blue = strong attention weight &nbsp;|&nbsp;
  light = weak or none
</div>
""")
display.display(header)

def update(target_pos):
    """Move the slider to pick a target position."""
    weights_row = get_attention_weights(target_pos)
    x_embed = model.embed(SEQ[:-1].unsqueeze(0)).detach()
    logits, _ = model(x_embed)
    predicted_char = i2c[int(logits.squeeze(0)[target_pos].argmax())]

    fig, ax = plt.subplots(figsize=(max(6, T * 0.5), 2.2))

    z = np.array(weights_row).reshape(1, -1)
    vmax = weights_row.max() if weights_row.max() > 0 else 1
    im = ax.imshow(z, cmap="Blues", vmin=0, vmax=vmax, aspect="auto", interpolation="nearest")

    ax.set_xticks(range(T))
    ax.set_xticklabels(labels, family="monospace", fontsize=11)
    ax.set_yticks([0])
    ax.set_yticklabels(["attention"], family="monospace", fontsize=10)

    rect = patches.Rectangle(
        (target_pos - 0.5, -0.5), 1, 1,
        linewidth=3, edgecolor="#E53935", facecolor="none"
    )
    ax.add_patch(rect)

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("weight", family="monospace", fontsize=10)

    ax.set_title(
        f"Predicting '{predicted_char}' at position {target_pos}  "
        f"('{TEXT[target_pos]}'→'{predicted_char}')  "
        f"— darker = more attention paid to this position",
        family="monospace", fontsize=11
    )
    ax.set_xlabel("input character position", family="monospace", fontsize=10)

    plt.tight_layout()
    plt.show()

interact(
    update,
    target_pos=widgets.IntSlider(
        min=0, max=T - 1, value=T // 2, step=1,
        description="target pos",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="520px"),
    ),
)

Compare this to the RNN heatmap from earlier. Instead of influence fading out the further back a character is, this model can jump directly back to whatever character actually matters, even if it is far away. Watch what happens right before predicting "cat", "sat", or "mat", the model tends to pay a lot of attention back to "the", since that is the character that reliably comes right before those words in this sentence.

## Multi Head Attention

Our tiny example above only computed one attention pattern at a time. Real models run several of these in parallel, each with its own learned query, key, and value weights. Each one of these is called a **head**. One head might end up focusing on grammar, like which word a pronoun refers to. Another might focus on which words are semantically related. The outputs of every head get combined back together at the end. This is called **multi head attention**, and it is one of the reasons transformers can pick up on so many different kinds of patterns in language at once.

In [ ]:
# @title
question = "Which of these best describes multi head attention?"
choices = [
    "Running the RNN forward and backward at the same time",
    "Giving the model two separate hidden states instead of one",
    "Training two different models and averaging their answers",
    "Running several attention calculations in parallel",
]
answer = choices[3]
hint = "Multi head attention does not change how memory works, it changes how many different attention patterns get computed at once"

question_label = widgets.Label(value=question)
choice_buttons = [widgets.Button(description=choice, layout=widgets.Layout(width="600px")) for choice in choices]
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if event.description == answer:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

for button in choice_buttons:
    button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    *choice_buttons,
    answer_result,
    hint_label,
]))

## Positional Encoding

There is one thing attention loses that an RNN never had a problem with. An RNN processes tokens one at a time in order, so the order is built right into how it works. Attention looks at every token all at once, in parallel, which is part of what makes it so much faster to train. But that also means attention on its own has no idea which token came first.

To fix this, before any attention happens, a **positional encoding** gets added directly onto each token's embedding. This is a set of numbers that is different for every position in the sequence, generated using sine and cosine waves of different frequencies. Take a look at a few of these waves below.

In [ ]:
# @title
positions = np.arange(0, 50)

for dim in range(4):
    freq = 1 / (10000 ** (dim / 4))
    plt.plot(positions, np.sin(positions * freq), label=f"dimension {dim}")

plt.grid()
plt.title("Positional Encoding")
plt.gca().set_xlabel("position in the sequence")
plt.gca().set_ylabel("encoding value")
plt.legend()
plt.show()

Every position in the sequence ends up with a unique combination of values across all these waves, kind of like how a specific time of day has a unique combination of positions for the hour hand and the minute hand. Adding this to a token's embedding gives the model a way to tell not just what a token is, but where it sits in the sequence.

In [ ]:
# @title
question = "Transformers naturally know the order of tokens in a sequence without any extra help, the same way an RNN does."
answer = "False"
hint = "Attention looks at every token in parallel, which is why positional encoding has to be added on separately"

question_label = widgets.Label(value=question)
true_button = widgets.Button(description="True")
false_button = widgets.Button(description="False")
answer_result = widgets.Label()
hint_label = widgets.Label()

def check_answer(event):
    if event.description == answer:
        answer_result.value = "✅ Correct!"
        hint_label.value = ""
    else:
        answer_result.value = "❌ Incorrect :("
        hint_label.value = hint

true_button.on_click(check_answer)
false_button.on_click(check_answer)

display.display(widgets.VBox([
    question_label,
    widgets.HBox([true_button, false_button]),
    answer_result,
    hint_label,
]))

## Putting It Together: The Transformer

In 2017 a group of researchers published a paper called Attention Is All You Need, and the name says exactly what they found. You do not need recurrence at all, you can build an entire model out of attention and one more piece you already know, the feedforward network from last lesson.

A single **transformer block** is just self attention followed by a feedforward network, applied to every token. The attention step lets tokens gather context from each other, and the feedforward step, the exact same kind of network you built for the weather example last lesson, gets applied to each token on its own to process what it just gathered. Real models stack dozens of these blocks on top of each other, each one refining the representation of every token a little further using richer and richer context.

The training goal is the same one Shannon and the n-gram researchers were chasing decades ago, predict the next token. The difference is a transformer is not limited to the last 2 to 5 tokens like an n-gram, and it is not limited to whatever survived being squeezed through a single hidden state like an RNN. Every token can look directly at every earlier token and decide for itself what actually matters.

This is the architecture behind GPT, Claude, and basically every large language model you have used. The recipe is not really a secret anymore, transformer blocks stacked on top of each other. What actually separates these models is scale, how many of these blocks are stacked, how large the vectors flowing through them are, and how much text they were trained on.

That scale is not free. The attention calculation compares every token in a sequence against every other token, so the amount of computation grows with the square of the sequence length. Doubling how much context a model can consider roughly quadruples the work involved. Training the largest models also takes an enormous amount of compute, electricity, and data, and that is only part of the cost. We are going to come back to some of those costs and tradeoffs at the end of this notebook.

# Check In - Attention and Transformers
We just learned how attention solves the bottleneck problem RNNs run into. Here are the key points you should know:

- Attention lets every token look directly at every earlier token, instead of relying on a single hidden state that gets rewritten at every step
- Query, key, and value vectors are used to score how relevant each earlier token is, and those scores get turned into weights using softmax
- Causal masking stops a model from attending to tokens that have not happened yet, which is required for predicting the next token
- Multi head attention runs several of these attention calculations in parallel so the model can pick up on different kinds of relationships at once
- Since attention processes all tokens in parallel, positional encoding has to be added so the model knows the order of the sequence
- A transformer block is just self attention followed by a feedforward network, and modern LLMs are built by stacking many of these blocks


# LLM Ethics - Group Discussion

Back near the start of this notebook we mentioned that as with all technology, these tools come at costs. Now that you understand roughly how attention and transformers work under the hood, it is worth pausing to talk through some of those costs and tradeoffs.

In your groups, or with your neighbors whoare are also at this point in the notebook, spend 5-10 minutes discussing the questions below. There is no single right answer to most of these, the goal is to think it through together and be ready to share one interesting point your group came up with.

## Impact on Society

- LLMs can now write code, essays, and answer factual questions faster than most people. What jobs or tasks do you think will change the most because of this in the next 10 years?
- Is it a problem if people rely on an LLM to think through a hard decision for them instead of thinking it through themselves? Does it matter what kind of decision it is?
- LLMs sometimes state incorrect information confidently, this is often called hallucination. What kinds of situations would this be dangerous in, and what kinds of situations would it not matter much?

## Ownership and Usage

- LLMs are trained on huge amounts of text scraped from the internet, much of it written by people who never agreed to have their writing used this way. Do you think that is fair? Does it depend on what the writing is used for?
- If an LLM generates an image, a song, or an essay in the style of a specific real artist or author, who owns that output? The person who typed the prompt, the company that built the model, or nobody?
- Should there be different rules for open source models that anyone can download and run, versus closed models that only run on a company's servers?

## Environmental and Social Cost

- Training and running large models requires enormous data centers, which use huge amounts of electricity and, in many cases, huge amounts of water for cooling. How should companies be required to disclose how much energy and water a model consumed to train?
- Data centers are often built in specific regions chosen for cheap land, cheap power, or lax regulation, and the communities nearby do not always share in the profits the data center generates. Who do you think should get a say in whether a data center gets built near where they live?
- If a wealthier country or company can afford to buy up water and power capacity for AI training, and that drives up costs or shortages for the surrounding community, who is responsible for that tradeoff?

## Jobs and the Economy

- If a company can replace a team of workers with an LLM subscription, the productivity gains and cost savings go mostly to the company's owners and shareholders, not to the workers who lost their jobs or to the public. Do you think that is a fair outcome? What, if anything, should change about that?
- Historically, new technology has destroyed some jobs and created others. Do you think LLMs will follow that same pattern, or is there something different about this technology that makes the comparison not hold up?
- A small number of companies control the most capable models and the data centers needed to train them. What happens to a society if the most economically important technology of an era is controlled by a handful of firms?
- If AI makes it possible to produce more goods and services with far fewer people working, who do you think should benefit from that gain, and how would that actually get decided?

## The Future

- Attention lets a model look at all of its previous context at once, but that gets more expensive the longer the context gets. What do you think happens to LLMs as computers get faster and cheaper?
- Do you think future LLMs will replace the need for people to learn how to code, or change what it means to know how to code?
- If an LLM starts making a lot of the small decisions in your life, what conversations, or what recommendations, would you want it to leave out of and let a person handle instead?

## Check-in
Check in with an instructor before moving on!

# Bonus - Build Your Own RNN
At this point you understand what an RNN does and why attention replaced it. Here is a very low stakes way to get your hands on one. We are going to build a tiny character level RNN, almost identical to the one used in the visualizations above. Poke at it to see how different changes affect the results.

In [ ]:
class CharDataset(Dataset):
    def __init__(self, text, seq_length=4):
        # Store the text, and figure out our vocabulary
        self.text = text
        self.seq_length = seq_length
        self.chars = sorted(set(text))
        self.c2i = {c: i for i, c in enumerate(self.chars)}
        self.i2c = {i: c for c, i in self.c2i.items()}

    def __len__(self):
        # Return the number of examples we can make from the text
        return len(self.text) - self.seq_length

    def __getitem__(self, index):
        # Return a chunk of characters, and the character that comes right after it
        input_chars = self.text[index:index + self.seq_length]
        target_char = self.text[index + self.seq_length]

        input_tensor = torch.tensor([self.c2i[c] for c in input_chars])
        target_tensor = torch.tensor(self.c2i[target_char])

        return input_tensor, target_tensor

In [ ]:
class CharPredictorRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=16):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_value):
        embedded = self.embed(input_value)
        rnn_output, hidden_state = self.rnn(embedded)

        # We only care about the prediction after the last character in the sequence
        last_output = rnn_output[:, -1, :]

        return self.output_layer(last_output)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

text = "the cat sat on the mat the cat sat on the mat the cat sat on the mat"
dataset = CharDataset(text, seq_length=4)

model = CharPredictorRNN(vocab_size=len(dataset.chars)).to(device)

learning_rate = 0.01
num_training_rounds = 300

error_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def training_loop(model, dataset, optimizer, error_fn):
    # Function which runs our model through a dataset and returns the error
    total_error = 0.0

    for input_data, expected_output in iter(dataset):
        # See what the model predicts
        prediction = model(input_data.unsqueeze(0).to(device))

        # Calculate the error (how close the prediction was to the correct result)
        error = error_fn(prediction, expected_output.unsqueeze(0).to(device))

        total_error = total_error + error

        # If an optimizer is specified then updates the model parameters using it
        if optimizer is not None:
            error.backward()
            optimizer.step()
            optimizer.zero_grad()

    return total_error

# Training loop
for training_round in range(num_training_rounds):
    model.train()

    total_error = training_loop(
        model=model,
        dataset=dataset,
        optimizer=optimizer,
        error_fn=error_fn,
      )

    # Print our progress every 50 training rounds
    if training_round % 50 == 0 or training_round == num_training_rounds - 1:
        print(f"Training round {training_round}, error {total_error}")

Now let's use our model! Feed it the start of the phrase and see what it predicts.

In [ ]:
model.eval()

start_text = "the c"
input_ids = torch.tensor([dataset.c2i[c] for c in start_text[-4:]]).unsqueeze(0).to(device)

prediction = model(input_ids)
predicted_char = dataset.i2c[int(prediction.argmax())]

print(f"After '{start_text}' the model predicts: '{predicted_char}'")

In [ ]:
# Generate several characters in a row by feeding the model's own predictions back in
generated = "the c"

for _ in range(20):
    input_ids = torch.tensor([dataset.c2i[c] for c in generated[-4:]]).unsqueeze(0).to(device)
    prediction = model(input_ids)
    predicted_char = dataset.i2c[int(prediction.argmax())]
    generated = generated + predicted_char

print(generated)

## You Try - Tweak Your RNN
Try changing a few things about the model above and see what happens:

- Change hidden_size in CharPredictorRNN, try a much smaller or much bigger number
- Change seq_length in CharDataset, this changes how many characters of memory the RNN gets to look at before predicting
- Change the text variable to a longer, less repetitive sentence and see if the model still learns it as well
- Change num_training_rounds and see how much training the model actually needs before the generated text stops looking random

Ask yourself, does the model do better or worse the longer or more repetitive the text is? Why do you think that is, based on what you learned about the bottleneck problem?

# Check In - Building an RNN
We just built and trained a tiny character level RNN from scratch. Here are the key points you should know:

- An RNN can be trained to predict the next character the same way a feedforward network predicts a probability, we just feed it a sequence instead of a single set of numbers
- The seq_length we choose limits how much memory the model gets to use when making a prediction
- A model this small, trained this briefly, is only ever going to be good at memorizing a short repetitive phrase, real language models need vastly more data and parameters to generalize